In [ ]:
!pip install opencv-python -q
import os, cv2, torch, numpy as np, pandas as pd
import torch.nn as nn
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as T
from sklearn.model_selection import train_test_split

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using:', DEVICE)

In [ ]:
DATA_DIR = '/kaggle/input/datasets/syxlicheng/heartdatabase/EchoNet-Dynamic'
VIDEO_DIR = os.path.join(DATA_DIR, 'Videos')
df = pd.read_csv(os.path.join(DATA_DIR, 'FileList.csv'))
df.head()

In [ ]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
print(len(train_df), len(val_df))

In [ ]:
class EchoDataset(Dataset):
    def __init__(self, video_dir, df, seq_len=16):
        self.video_dir = video_dir
        self.df = df.reset_index(drop=True)
        self.seq_len = seq_len
        self.transform = T.Compose([
            T.ToTensor(),
            T.Resize((224,224)),
        ])

    def load_video(self, path):
        cap = cv2.VideoCapture(path)
        frames = []
        while len(frames) < self.seq_len:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = self.transform(frame)
            frames.append(frame)
        cap.release()
        while len(frames) < self.seq_len:
            frames.append(frames[-1])
        return torch.stack(frames)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.video_dir, row['FileName'] + '.avi')
        video = self.load_video(path)
        label = torch.tensor(row['EF'] > 50).long()
        return video, label

    def __len__(self):
        return len(self.df)

In [ ]:
train_dataset = EchoDataset(VIDEO_DIR, train_df)
val_dataset = EchoDataset(VIDEO_DIR, val_df)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4)

In [ ]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet18(weights='IMAGENET1K_V1')
        self.encoder = nn.Sequential(*list(base.children())[:-1])
        self.lstm = nn.LSTM(512, 256, batch_first=True)
        self.fc = nn.Linear(256, 2)

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B*T, C, H, W)
        feats = self.encoder(x).view(B, T, -1)
        out, _ = self.lstm(feats)
        return self.fc(out[:, -1])

In [ ]:
model = Model().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

In [ ]:
class EarlyStopping:
    def __init__(self, patience=3, delta=0, path="best_model.pth"):
        self.patience = patience
        self.delta = delta
        self.best_score = None
        self.counter = 0
        self.early_stop = False
        self.path = path

    def __call__(self, val_acc, model):
        score = val_acc

        if self.best_score is None:
            self.best_score = score
            self.save_model(model)

        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter}/{self.patience}")

            if self.counter >= self.patience:
                self.early_stop = True

        else:
            self.best_score = score
            self.save_model(model)
            self.counter = 0

    def save_model(self, model):
        torch.save(model.state_dict(), self.path)
        print("✅ Model improved. Saved.")

In [ ]:
early_stopping = EarlyStopping(patience=3)

EPOCHS = 20

for epoch in range(EPOCHS):
    loss = train_one_epoch(model, train_loader)
    acc = validate(model, val_loader)

    print(f"Epoch {epoch}: Loss={loss:.4f}, Acc={acc:.4f}")

    early_stopping(acc, model)

    if early_stopping.early_stop:
        print("⛔ Early stopping triggered")
        break

In [ ]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0
    for x, y in tqdm(loader):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
def validate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            pred = torch.argmax(out, dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total

In [ ]:
for epoch in range(3):
    loss = train_one_epoch(model, train_loader)
    acc = validate(model, val_loader)
    print(epoch, loss, acc)

In [ ]:
# ===== FINAL FIXED GRAD-CAM (STRONG + STABLE) =====

class GradCAM:
    def __init__(self, model):
        self.model = model
        self.gradients = None
        self.activations = None

        def f_hook(m, i, o):
            self.activations = o

        def b_hook(m, gi, go):
            self.gradients = go[0]

        # 🔥 IMPORTANT: use second last layer (better spatial info)
        target_layer = self.model.encoder[-2]

        target_layer.register_forward_hook(f_hook)
        target_layer.register_full_backward_hook(b_hook)

    def generate(self, x, class_idx):
        self.model.zero_grad()

        out = self.model(x)
        out[:, class_idx].backward()

        grads = self.gradients
        acts = self.activations

        weights = torch.mean(grads, dim=(2,3))

        cam = torch.zeros(acts.shape[2:]).to(x.device)

        for i, w in enumerate(weights[0]):
            cam += w * acts[0, i]

        cam = torch.relu(cam)

        # 🔥 IMPROVED NORMALIZATION
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)

        return cam.detach().cpu().numpy()


# ===== RUN (FIX FOR LSTM) =====
model.train()   # 🔥 REQUIRED

gradcam = GradCAM(model)

video, _ = val_dataset[0]
video = video.unsqueeze(0).to(DEVICE)

pred = torch.argmax(model(video), dim=1).item()
cam = gradcam.generate(video, pred)

model.eval()   # optional


# ===== VISUALIZATION (STRONG COLORS) =====
frame = video[0,8].cpu().permute(1,2,0).numpy()   # 🔥 use middle frame
frame = (frame * 255).astype("uint8")

cam = cv2.resize(cam, (224,224))

# 🔥 better colormap
heatmap = cv2.applyColorMap(np.uint8(255*cam), cv2.COLORMAP_HOT)

overlay = cv2.addWeighted(frame, 0.6, heatmap, 0.5, 0)

plt.imshow(overlay)
plt.axis("off")